# 01 — What Are Structured Outputs? — Practical Examples

**Covers**: Unstructured vs structured comparison, three categories in action, Pydantic basics, real-world extraction

In [ ]:
import os, json
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. The Problem — Unstructured Output Chaos

In [ ]:
# Ask the same question 3 times — watch the format chaos
prompt = """Extract the customer name and order amount from this email:
'Hi, I'm Sarah Johnson. I'd like to confirm my order of $312.50.'
Return as JSON."""

for i in range(3):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.9  # High temp to show variance
    )
    print(f"Run {i+1}: {r.choices[0].message.content!r}")
    print()

## 2. Category 1 — Prompt-Based (No Guarantee)

In [ ]:
# Prompt engineering approach — ask nicely
response = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": """Extract from this: 'Order #A-1024 placed by John Doe for $789.99'
        Return ONLY JSON: {\"customer\": str, \"order_id\": str, \"amount\": float}"""
    }]
)
raw = response.choices[0].message.content
print(f"Raw output: {raw!r}")

# Risky parse
try:
    # Strip potential markdown fences
    cleaned = raw.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
    data = json.loads(cleaned)
    print(f"Parsed: {data}")
except json.JSONDecodeError as e:
    print(f"PARSE FAILED: {e}")

## 3. Category 2 — JSON Mode (Valid JSON, Schema Not Enforced)

In [ ]:
# JSON mode - output is always valid JSON, but schema is up to the LLM
response = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},  # JSON mode!
    messages=[
        {"role": "system", "content": "You are a data extractor. Always return JSON."},
        {"role": "user",   "content": "Extract from: 'Order #A-1024 placed by John Doe for $789.99'. Fields: customer, order_id, amount"}
    ]
)
raw = response.choices[0].message.content
data = json.loads(raw)  # Safe! Always valid JSON in JSON mode
print(f"Type of raw: {type(raw).__name__}")
print(f"Parsed object: {data}")
print(f"Type of amount: {type(data.get('amount', data.get('order_amount', '?'))).__name__}") # May vary!

## 4. Category 3 — Strict Structured Outputs (Schema Guaranteed)

In [ ]:
# Define the EXACT schema with Pydantic
class OrderExtraction(BaseModel):
    customer_name: str
    order_id: str
    amount: float
    currency: str = "USD"  # With default

# Use .parse() for strict structured output
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": "Extract from: 'Order #A-1024 placed by John Doe for $789.99'"
    }],
    response_format=OrderExtraction
)

result = response.choices[0].message.parsed
print(f"Type: {type(result).__name__}")
print(f"customer_name: {result.customer_name!r}  (type: {type(result.customer_name).__name__})")
print(f"order_id:      {result.order_id!r}")
print(f"amount:        {result.amount}  (type: {type(result.amount).__name__})")
print(f"currency:      {result.currency!r}")

## 5. Comparing All Three Side by Side

In [ ]:
import time

EMAIL = "Customer Alice Wang placed order #ZX-2048 for 3 items totaling $1,245.00."

# --- Approach 1: Prompt-only ---
start = time.perf_counter()
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract JSON with keys: customer_name, order_id, amount.\n{EMAIL}"}]
)
prompt_time = time.perf_counter() - start
prompt_out = r1.choices[0].message.content

# --- Approach 2: JSON Mode ---
start = time.perf_counter()
r2 = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": "Extract data as JSON."},
        {"role": "user",   "content": f"Fields: customer_name, order_id, amount.\n{EMAIL}"}
    ]
)
json_time = time.perf_counter() - start
json_out = json.loads(r2.choices[0].message.content)

# --- Approach 3: Strict Structured Output ---
class OrderInfo(BaseModel):
    customer_name: str
    order_id: str
    amount: float

start = time.perf_counter()
r3 = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": EMAIL}],
    response_format=OrderInfo
)
strict_time = time.perf_counter() - start
strict_out = r3.choices[0].message.parsed

# --- Summary ---
print("\n" + "="*55)
print(f"{'Method':<20} {'Schema Safe?':<14} {'Time':<10} {'Types Correct?'}")
print("="*55)
print(f"{'Prompt-based':<20} {'❌ No':<14} {prompt_time:.2f}s      {'⚠️  Maybe'}")
print(f"{'JSON Mode':<20} {'⚠️  Partial':<14} {json_time:.2f}s      {'⚠️  Maybe'}")
print(f"{'Strict Struct.':<20} {'✅ Yes':<14} {strict_time:.2f}s      {'✅ Yes'}")
print("="*55)
print(f"\nStrict result → customer={strict_out.customer_name!r}, amount={strict_out.amount} ({type(strict_out.amount).__name__})")

## 6. Nested Structured Outputs — Complex Schemas

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Nested schema — multi-level structure
class Address(BaseModel):
    street: str
    city: str
    country: str

class CustomerProfile(BaseModel):
    full_name: str
    email: str
    tier: Literal["bronze", "silver", "gold", "platinum"]
    lifetime_value: float = Field(description="Total amount spent in USD")
    shipping_address: Address
    tags: list[str] = Field(default_factory=list)

bio_text = """
Jane Smith (jane@example.com) is a Gold tier member who has spent $4,320 total.
She ships to 42 Baker Street, London, UK.
She's flagged as: loyal, high-value, international.
"""

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": f"Extract customer info:\n{bio_text}"}],
    response_format=CustomerProfile
)

profile = response.choices[0].message.parsed
rprint(profile.model_dump())  # Pretty print the full nested object
print(f"\nAddress city: {profile.shipping_address.city}")
print(f"Tags: {profile.tags}")

## 7. Classification with Literals — Enum-Like Structured Output

In [ ]:
from typing import Literal

class SupportTicketClassification(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    category: Literal["billing", "technical", "shipping", "general", "refund"]
    priority: Literal["low", "medium", "high", "critical"]
    requires_human: bool
    one_line_summary: str

tickets = [
    "My package hasn't arrived in 3 weeks and I need it for my wedding tomorrow!",
    "Just wanted to say the new app update is fantastic. Love the dark mode!",
    "Can you explain what the different subscription tiers include?"
]

for ticket in tickets:
    r = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a support ticket classifier."},
            {"role": "user",   "content": ticket}
        ],
        response_format=SupportTicketClassification
    )
    t = r.choices[0].message.parsed
    print(f"Ticket:  {ticket[:60]}...")
    print(f"  sentiment={t.sentiment}, category={t.category}, priority={t.priority}, human={t.requires_human}")
    print(f"  summary: {t.one_line_summary}")
    print()

## 📌 Summary

| Approach | Valid JSON? | Schema Enforced? | Type Safe? | Best For |
|---|---|---|---|---|
| Prompt-based | ❌ | ❌ | ❌ | Prototyping only |
| JSON Mode | ✅ | ❌ | ❌ | Simple use cases |
| Strict Structured | ✅ | ✅ | ✅ | Production pipelines |